In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import calinski_harabasz_score
import matplotlib.pyplot as plt
import ancestree

rules = {
    "raw data": [None],
    "scaling": ["raw data"],
    "embedding": ["scaling"],
    "clustering": ["embedding"]
}
triggers = ["scaling", "embedding", "clustering"]

store = ancestree.LineageStore("ML_pipeline", rules=rules, gen_triggers=triggers)

/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
scalers = {"robust": RobustScaler(), "standard":StandardScaler()}
embedders = ["umap", "tsne", "pca"]
clusterers = ["kmeans", "hdbscan"]

with store.create_node("raw data") as root_node:
    iris = load_iris()
    df_raw = pd.DataFrame(iris.data, columns = iris.feature_names)
    df_raw.to_csv(root_node/"data.csv", index=False)

    # Scaling
    for s_name, s_obj in scalers.items():
        with store.create_node("scaling", parent=root_node) as s_node:
            scaled_data = s_obj.fit_transform(df_raw)
            pd.DataFrame(scaled_data).to_csv(s_node/"scaled.csv", index=False)

            # Dim reduction
            for e_type in embedders:
                with store.create_node("embedding", parent=s_node) as e_node:
                    if e_type == "pca":
                        embedder = PCA(n_components=2)
                    elif e_type == "tsne":
                        embedder = TSNE(n_components=2, perplexity=30)
                    elif e_type == "umap":
                        embedder = umap.UMAP(n_components=2)
                    
                    embedding = embedder.fit_transform(scaled_data)

                    np.save(e_node/"map.npy", embedding)
                    e_node.add_meta('method', e_type, 'text', 'Method')

                    # Clustering
                    for c_type in clusterers:
                        with store.create_node("clustering", parent=e_node) as c_node:
                            if c_type == "kmeans":
                                model = KMeans(n_clusters=3, n_init="auto")
                                labels = model.fit_predict(embedding)
                            elif c_type == "hdbscan":
                                model = HDBSCAN(min_cluster_size=5)
                                labels = model.fit_predict(embedding)
                            
                            score = calinski_harabasz_score(embedding, labels)
                            c_node.add_meta('CH_score', score, 'text', 'Metrics')

                            plt.figure(figsize=(6,4))
                            plt.scatter(embedding[:, 0], embedding[:,1], c=labels, cmap='turbo')
                            plt.savefig(c_node/"clustering_img.png")
                            plt.close()
                            c_node.add_meta('Final Results', str(c_node/"clustering_img.png"), 'image', 'Results')

/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default

In [3]:
c_node/"img"

PosixPath('ML_pipeline/42ee0c5e/img')

In [7]:
import ancestree
store = ancestree.LineageStore(root='ML_pipeline')
store.generate_web_graph()

Graph generated at ML_pipeline/interactive_pipeline.html


In [4]:
crappy_nodes = store.find_node(CH_score = lambda x: x<1000)
crappy_nodes

[Node = 263c807c, path = 263c807c, generation = 3,
 Node = 248fc5bf, path = 248fc5bf, generation = 3,
 Node = 9eadaa47, path = 9eadaa47, generation = 3,
 Node = 49b87315, path = 49b87315, generation = 3,
 Node = 73a51600, path = 73a51600, generation = 3,
 Node = 2d8927e2, path = 2d8927e2, generation = 3]

In [6]:
for node in crappy_nodes:
    store.prune(node, recursive=False, dry_run=False)

In [7]:
import ancestree
store = ancestree.LineageStore(root='ML_pipeline')
store.generate_web_graph()

Graph generated at ML_pipeline/interactive_pipeline.html


In [8]:
nodes_of_interest = store.find_node(method='pca', step_type='embedding')
print(nodes_of_interest)



[Node = aa1eaa97, path = aa1eaa97, generation = 2, Node = 7e70194a, path = 7e70194a, generation = 2]


In [9]:
store.generate_web_graph()

Graph generated at ML_pipeline/interactive_pipeline.html


In [11]:
node = store.get_node('0d33f882')

node.artifacts('*.npy')


[PosixPath('0d33f882/map.npy')]

In [15]:
lineage = store.get_lineage('0d33f882')
# print(lineage)
for nodes in lineage:
    print(nodes.node_id)
    print(nodes.artifacts('*.csv'))

221b9646
[PosixPath('221b9646/data.csv')]
71a4cc92
[PosixPath('71a4cc92/scaled.csv')]
0d33f882
[]


In [17]:
import ancestree

store = ancestree.LineageStore('ML_pipeline')

print(store.find_node(step_type='embedding'))

print(store.get_node(node='0d33f882'))

print(store.get_lineage(node = '0d33f882'))

print(store.find_in_lineage(node='0d33f882', step_type='embedding'))

print(store.get_most_recent_node())

print(store.get_child_nodes(node='0d33f882'))

print(store.from_parent(node = '0d33f882', filename='*.csv'))



[Node = 56a8a443, path = 56a8a443, generation = 2, Node = 43048584, path = 43048584, generation = 2, Node = 3c0238e8, path = 3c0238e8, generation = 2, Node = 793c2bf8, path = 793c2bf8, generation = 2, Node = f1d0193a, path = f1d0193a, generation = 2, Node = c9785b7d, path = c9785b7d, generation = 2, Node = 97a26592, path = 97a26592, generation = 2, Node = 3366df4e, path = 3366df4e, generation = 2, Node = 09cef53f, path = 09cef53f, generation = 2, Node = ec8cab2f, path = ec8cab2f, generation = 2, Node = 0d33f882, path = 0d33f882, generation = 2, Node = dfe9a140, path = dfe9a140, generation = 2, Node = aa1eaa97, path = aa1eaa97, generation = 2, Node = 265564ae, path = 265564ae, generation = 2, Node = 7e70194a, path = 7e70194a, generation = 2, Node = 9d3a63ed, path = 9d3a63ed, generation = 2, Node = e52f039a, path = e52f039a, generation = 2, Node = e644d866, path = e644d866, generation = 2]
Node = 0d33f882, path = 0d33f882, generation = 2
[Node = 221b9646, path = 221b9646, generation = 0,